# FinOps & DAG Analysis (scratch)

Working notebook for the final report: real run timings from `pipeline_execution_logs`, cloud-credit consumption from `system.billing.usage`, and inputs for the Spark DAG analysis. Nothing here writes to pipeline tables — safe to delete once the report is done.

In [0]:
%run ./00_config_and_logging

In [0]:
from pyspark.sql import functions as F

# Per-step durations of every successful load today (Oct 24, 2026):
# full rebuild, gold builds, asset swap, and the incremental batch chain.
steps = (spark.table(LOG_TABLE)
    .where((F.col("status") == "Success") & (F.col("start_time") >= "2026-10-24"))
    .select("layer", "load_type", "source_param", "target_table",
            F.round(F.unix_timestamp("end_time") - F.unix_timestamp("start_time"), 1).alias("run_seconds"),
            "rows_inserted", "rows_updated",
            F.date_format("start_time", "HH:mm:ss").alias("started_at"))
    .orderBy("start_time"))
display(steps)

print("Total pipeline compute-seconds logged today:",
      steps.agg(F.round(F.sum("run_seconds"), 1).alias("s")).first()[0])

In [0]:
from pyspark.sql import functions as F

agg = (steps.groupBy("layer")
       .agg(F.count("*").alias("steps"),
            F.round(F.sum("run_seconds"), 1).alias("total_seconds"),
            F.round(F.avg("run_seconds"), 1).alias("avg_seconds"),
            F.round(F.max("run_seconds"), 1).alias("max_seconds"))
       .orderBy(F.desc("total_seconds")))
display(agg)

# Focus: the incremental batch chain (Raw->Bronze incremental + the Silver/Gold
# MERGEs that followed it). The Bronze incremental rows of today are the batch
# 2026-10-13_060009 load; silver 'merge' rows that day are its Silver MERGE.
incr_chain = steps.where(F.col("load_type") == "incremental")
display(incr_chain)
print("Bronze incremental load seconds:", incr_chain.agg(F.sum("run_seconds")).first()[0])

In [0]:
try:
    billing = spark.sql("""
        SELECT u.sku_name, u.billing_origin_product,
               round(sum(u.usage_quantity), 3) AS dbus,
               round(sum(u.usage_quantity * pricing.`default`), 4) AS est_usd
        FROM system.billing.usage u
        LEFT JOIN system.billing.list_prices pricing
          ON u.sku_name = pricing.sku_name
         AND pricing.price_start_time <= u.usage_start_time
         AND (pricing.price_end_time >= u.usage_start_time OR pricing.price_end_time IS NULL)
         AND pricing.currency_code = 'USD'
        WHERE u.usage_date >= current_date() - INTERVAL 7 DAYS
        GROUP BY u.sku_name, u.billing_origin_product
        HAVING sum(u.usage_quantity) != 0
        ORDER BY dbus DESC
    """)
    display(billing)
    tot = billing.agg(F.round(F.sum("dbus"), 1).alias("dbus_7d"),
                      F.round(F.sum("est_usd"), 2).alias("usd_7d")).first()
    print(f"Last 7 days: {tot['dbus_7d']} DBUs, ~${tot['usd_7d']}")
except Exception as e:
    print("Billing table not accessible in this workspace:", str(e)[:300])

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType
import re

# --- Replicate the fact build (03_silver_to_gold cell 10, pre-MERGE) to capture
# --- its physical plan and cardinalities. Nothing is written to any table.

def _parse(v):
    if v is None:
        return []
    parts = []
    for seg in str(v).split("."):
        m = re.match(r"^(\d+)", seg.strip())
        parts.append((int(m.group(1)) if m else 0, seg.strip()))
    return parts

def _cmp(a, b):
    n = max(len(a), len(b))
    a = a + [(0, "")] * (n - len(a))
    b = b + [(0, "")] * (n - len(b))
    for sa, sb in zip(a, b):
        if sa != sb:
            return -1 if sa < sb else 1
    return 0

def version_matches(asset_ver, cpe_ver, s_exc, s_inc, e_exc, e_inc):
    if asset_ver is None:
        return False
    a = _parse(asset_ver)
    if cpe_ver not in (None, "", "*") and _cmp(a, _parse(cpe_ver)) != 0:
        return False
    if s_exc and not _cmp(a, _parse(s_exc)) > 0:
        return False
    if s_inc and _cmp(a, _parse(s_inc)) < 0:
        return False
    if e_exc and not _cmp(a, _parse(e_exc)) < 0:
        return False
    if e_inc and _cmp(a, _parse(e_inc)) > 0:
        return False
    return True

version_matches_udf = udf(version_matches, BooleanType())

cpe_rows = (spark.table(BRONZE_NVD)
    .select(F.col("id").alias("cve_id"), "configurations")
    .dropDuplicates(["cve_id"])
    .select("cve_id", F.explode_outer("configurations").alias("conf"))
    .select("cve_id", F.explode_outer("conf.nodes").alias("node"))
    .select("cve_id", F.explode_outer("node.cpeMatch").alias("cm"))
    .where(F.col("cm.criteria").isNotNull() & (F.col("cm.vulnerable") == True))
    .select("cve_id",
            F.split(F.col("cm.criteria"), ":").alias("parts"),
            F.col("cm.versionStartExcluding").alias("vs_exc"),
            F.col("cm.versionStartIncluding").alias("vs_inc"),
            F.col("cm.versionEndExcluding").alias("ve_exc"),
            F.col("cm.versionEndIncluding").alias("ve_inc"))
    .select("cve_id",
            F.col("parts")[3].alias("cpe_vendor"),
            F.col("parts")[4].alias("cpe_product"),
            F.col("parts")[5].alias("cpe_version"),
            "vs_exc", "vs_inc", "ve_exc", "ve_inc"))

fact = (cpe_rows
    .join(spark.table(GOLD_DIM_SOFTWARE).select("software_key", "software_name", "cpe_vendor", "cpe_product"),
          on=["cpe_vendor", "cpe_product"])
    .join(spark.table(GOLD_DIM_ASSET).select("asset_id", "software_name", "software_version"),
          on="software_name")
    .where(version_matches_udf(F.col("software_version"), F.col("cpe_version"),
                               F.col("vs_exc"), F.col("vs_inc"), F.col("ve_exc"), F.col("ve_inc")))
    .join(spark.table(SILVER_UNIFIED).select(
            "cve_id", "published_date", "kev_date_added",
            "cvss_base_score", "epss_score", "epss_percentile", "is_in_kev"),
          on="cve_id")
    .select(
        "asset_id", "cve_id",
        F.sha2(F.trim(F.col("cve_id")), 256).alias("cve_key"),
        "software_key",
        F.when(F.col("cvss_base_score") >= 9.0, F.lit(4))
         .when(F.col("cvss_base_score") >= 7.0, F.lit(3))
         .when(F.col("cvss_base_score") >= 4.0, F.lit(2))
         .when(F.col("cvss_base_score") >= 0.1, F.lit(1))
         .otherwise(F.lit(0)).alias("severity_key"),
        F.date_format("published_date", "yyyyMMdd").cast("int").alias("published_date_key"),
        F.when(F.col("kev_date_added").isNotNull(),
               F.date_format("kev_date_added", "yyyyMMdd").cast("int")).alias("kev_date_key"),
        "cvss_base_score", "epss_score", "epss_percentile", "is_in_kev",
        F.when(F.col("is_in_kev") == True, F.lit(1))
         .when(F.col("epss_score") >= 0.5, F.lit(2))
         .when(F.col("cvss_base_score") >= 9.0, F.lit(3))
         .otherwise(F.lit(4)).alias("priority_rank"))
    .dropDuplicates(["asset_id", "cve_key"]))

print("== Cardinalities ==")
print("Bronze NVD distinct CVEs:", spark.table(BRONZE_NVD).select("id").distinct().count())
print("Exploded CPE criteria rows (cve x cpeMatch):", cpe_rows.count())
print("Fact rows after joins + version filter + dedup:", fact.count())
print()
print("== Physical plan (DAG) ==")
fact.explain("formatted")

In [0]:
from pyspark.sql import Window

# DETERMINISM CHECK: cell 10 uses dropDuplicates(["cve_id"]) which compiles to
# first() — arbitrary pick among differing bronze rows (full vs incremental
# configurations). Fresh count came out 634 vs 1,031 rows in the target.
# Latest-wins window (same pattern as dim_asset) should make it deterministic.
nvd_latest = (spark.table(BRONZE_NVD)
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("id").orderBy(F.col("load_timestamp").desc())))
    .where(F.col("rn") == 1).drop("rn")
    .select(F.col("id").alias("cve_id"), "configurations"))

print("Bronze NVD rows:", spark.table(BRONZE_NVD).count(),
      "| distinct CVEs:", nvd_latest.count())

cpe_rows_det = (nvd_latest
    .select("cve_id", F.explode_outer("configurations").alias("conf"))
    .select("cve_id", F.explode_outer("conf.nodes").alias("node"))
    .select("cve_id", F.explode_outer("node.cpeMatch").alias("cm"))
    .where(F.col("cm.criteria").isNotNull() & (F.col("cm.vulnerable") == True))
    .select("cve_id",
            F.split(F.col("cm.criteria"), ":").alias("parts"),
            F.col("cm.versionStartExcluding").alias("vs_exc"),
            F.col("cm.versionStartIncluding").alias("vs_inc"),
            F.col("cm.versionEndExcluding").alias("ve_exc"),
            F.col("cm.versionEndIncluding").alias("ve_inc"))
    .select("cve_id",
            F.col("parts")[3].alias("cpe_vendor"),
            F.col("parts")[4].alias("cpe_product"),
            F.col("parts")[5].alias("cpe_version"),
            "vs_exc", "vs_inc", "ve_exc", "ve_inc"))

fact_det = (cpe_rows_det
    .join(spark.table(GOLD_DIM_SOFTWARE).select("software_key", "software_name", "cpe_vendor", "cpe_product"),
          on=["cpe_vendor", "cpe_product"])
    .join(spark.table(GOLD_DIM_ASSET).select("asset_id", "software_name", "software_version"),
          on="software_name")
    .where(version_matches_udf(F.col("software_version"), F.col("cpe_version"),
                               F.col("vs_exc"), F.col("vs_inc"), F.col("ve_exc"), F.col("ve_inc")))
    .join(spark.table(SILVER_UNIFIED).select(
            "cve_id", "published_date", "kev_date_added",
            "cvss_base_score", "epss_score", "epss_percentile", "is_in_kev"),
          on="cve_id")
    .select("asset_id",
        F.sha2(F.trim(F.col("cve_id")), 256).alias("cve_key"),
        F.when(F.col("is_in_kev") == True, F.lit(1))
         .when(F.col("epss_score") >= 0.5, F.lit(2))
         .when(F.col("cvss_base_score") >= 9.0, F.lit(3))
         .otherwise(F.lit(4)).alias("priority_rank"))
    .dropDuplicates(["asset_id", "cve_key"]))

print("Deterministic (latest-wins) fact count:", fact_det.count())

# Compare with the target for drift
tgt = spark.table("workspace.cyber_threat.gold_fact_vulnerability").select("asset_id", "cve_key").distinct()
print("gold_fact_vulnerability rows in target:", tgt.count())

In [0]:
# WHERE did the ~390 missing exposures go? Per-asset target vs fresh build.
tgt_counts = (spark.table("workspace.cyber_threat.gold_fact_vulnerability")
              .groupBy("asset_id").agg(F.count("*").alias("in_target")))
fresh_counts = fact_det.groupBy("asset_id").agg(F.count("*").alias("fresh"))

comparison = (tgt_counts.join(fresh_counts, "asset_id", "full_outer")
    .na.fill(0)
    .withColumn("delta", F.col("fresh") - F.col("in_target"))
    .orderBy(F.col("delta")))
display(comparison)

# And the asset attributes in play: does dim_asset still have the vulnerable versions?
display(spark.table("workspace.cyber_threat.gold_dim_asset")
         .select("asset_id", "software_name", "software_version")
         .orderBy("software_name", "asset_id"))

In [0]:
# Root cause hunt: look at bronze rows behind exposures that are in the target
# but missing from the fresh build (MySQL examples).
missing = (spark.table("workspace.cyber_threat.gold_fact_vulnerability")
    .select("asset_id", "cve_id", "cve_key")
    .join(fact_det, on=["asset_id", "cve_key"], how="left_anti"))

print("Missing from fresh build:", missing.count(),
      "| MySQL-02:", missing.where("asset_id = 'AST-MYSQL-02'").count())

samples = missing.where("asset_id = 'AST-MYSQL-02'").limit(3).collect()
for r in samples:
    cve = r["cve_id"]
    rows = (spark.table(BRONZE_NVD).where(F.col("id") == cve)
            .select("id", "load_timestamp", "load_type", "source_file",
                    F.size("configurations").alias("n_configs"))
            .orderBy("load_timestamp")).toPandas()
    print(f"\n=== {cve} ({len(rows)} bronze rows) ===")
    print(rows.to_string(index=False))

In [0]:
# Micro-test: why does CVE-2021-2030 (in target for AST-MYSQL-02) not match now?
cve = "CVE-2021-2030"
cms = (spark.table(BRONZE_NVD).where(F.col("id") == cve)
    .select(F.explode_outer("configurations").alias("conf"))
    .select(F.explode_outer("conf.nodes").alias("node"))
    .select(F.explode_outer("node.cpeMatch").alias("cm"))
    .where(F.col("cm.criteria").isNotNull())
    .select("cm").toPandas())

import pandas as pd
pd.set_option("display.width", 200)
print(f"{cve} cpeMatch entries:")
print(cms.to_string(index=False))

asset_ver = "8.0.0"
print(f"\nUDF results for software_version='{asset_ver}':")
for _, row in cms.iterrows():
    cm = row["cm"]
    res = version_matches(asset_ver, cm["criteria"].split(":")[5],
                          cm["versionStartExcluding"], cm["versionStartIncluding"],
                          cm["versionEndExcluding"], cm["versionEndIncluding"])
    print(f"  {cm['criteria']:55s} vulnerable={cm['vulnerable']} -> {res}")

In [0]:
# CORRECTED check: the pipeline's cell 10 already has the right bounds logic
# (_cmp < 0 for s_inc, _cmp > 0 for e_inc). My earlier scratch replica used a
# stale copy with inverted s_inc/e_inc checks -> the 634 was MY typo, not the
# pipeline's. Verify with correct logic: expect ~1,031 and zero drift.
def version_matches_ok(asset_ver, cpe_ver, s_exc, s_inc, e_exc, e_inc):
    if asset_ver is None:
        return False
    a = _parse(asset_ver)
    if cpe_ver not in (None, "", "*") and _cmp(a, _parse(cpe_ver)) != 0:
        return False
    if s_exc and not _cmp(a, _parse(s_exc)) > 0:
        return False
    if s_inc and _cmp(a, _parse(s_inc)) < 0:
        return False
    if e_exc and not _cmp(a, _parse(e_exc)) < 0:
        return False
    if e_inc and _cmp(a, _parse(e_inc)) > 0:
        return False
    return True

version_matches_ok_udf = udf(version_matches_ok, BooleanType())

fact_ok = (cpe_rows_det
    .join(spark.table(GOLD_DIM_SOFTWARE).select("software_key", "software_name", "cpe_vendor", "cpe_product"),
          on=["cpe_vendor", "cpe_product"])
    .join(spark.table(GOLD_DIM_ASSET).select("asset_id", "software_name", "software_version"),
          on="software_name")
    .where(version_matches_ok_udf(F.col("software_version"), F.col("cpe_version"),
                                  F.col("vs_exc"), F.col("vs_inc"), F.col("ve_exc"), F.col("ve_inc")))
    .join(spark.table(SILVER_UNIFIED).select("cve_id"), on="cve_id")
    .select("asset_id", F.sha2(F.trim(F.col("cve_id")), 256).alias("cve_key"))
    .dropDuplicates(["asset_id", "cve_key"]))

print("Correct-logic fact count:", fact_ok.count())

ok_pairs = fact_ok
stale = (spark.table("workspace.cyber_threat.gold_fact_vulnerability")
         .select("asset_id", "cve_key")
         .join(ok_pairs, on=["asset_id", "cve_key"], how="left_anti"))
print("Rows in target NOT reproduced by correct build (stale/bogus):", stale.count())

In [0]:
# Single-action job for the report screenshot: runs the FULL fact-build pipeline
# (scan -> explode x3 -> cve-dedup shuffle -> broadcast joins -> UDF version
# check -> silver shuffle join -> final dedup) in ONE job, minus the MERGE
# write, so the query-profile DAG is compact and screenshotable.
print("Fact build row count:", fact.count())

In [0]:
# Exact column names of all six gold tables (for Power BI DAX references).
for t in ["gold_fact_vulnerability", "gold_dim_asset", "gold_dim_cve",
          "gold_dim_software", "gold_dim_severity", "gold_dim_date"]:
    cols = [f"{f.name} ({f.dataType.simpleString()})" for f in spark.table(f"workspace.cyber_threat.{t}").schema.fields]
    print(f"{t}:")
    for c in cols:
        print(f"  {c}")

In [0]:
from pyspark.sql import functions as F

# Distinct values of the categorical fields the DAX measures filter on.
display(spark.table("workspace.cyber_threat.gold_dim_cve")
        .where(F.col("known_ransomware_use").isNotNull())
        .groupBy("known_ransomware_use").count())
display(spark.table("workspace.cyber_threat.gold_dim_severity")
        .select("severity_key", "severity_name", "severity_rank"))

In [0]:
# Exact column names of the audit/log table (for the Power BI 'Pipeline Operations' page).
for f in spark.table(LOG_TABLE).schema.fields:
    print(f"{f.name} ({f.dataType.simpleString()})")
print()
display(spark.table(LOG_TABLE).groupBy("status").count())

In [0]:
# Ground truth for Power BI: composite Risk Score per hostname (same formula
# as the DAX measure), so the dashboard values can be verified.
risk = (spark.table("workspace.cyber_threat.gold_fact_vulnerability")
    .join(spark.table("workspace.cyber_threat.gold_dim_asset"), on="asset_id")
    .withColumn("risk_points",
        F.when(F.col("is_in_kev") == True, F.lit(15.0))
         .otherwise(F.coalesce(F.col("epss_score"), F.lit(0.0)) * 4
                    + F.coalesce(F.col("cvss_base_score"), F.lit(0.0))))
    .groupBy("hostname")
    .agg(F.round(F.sum("risk_points"), 1).alias("risk_score"),
         F.count("*").alias("exposures"),
         F.sum(F.when(F.col("is_in_kev") == True, 1).otherwise(0)).alias("kev"))
    .orderBy(F.desc("risk_score")))
display(risk)

print("TOTAL risk score:", risk.agg(F.round(F.sum("risk_score"), 1)).first()[0])
print("KEV exposures:", risk.agg(F.sum("kev")).first()[0])
print("Avg risk per exposure:",
      risk.agg(F.round(F.sum("risk_score") / F.sum("exposures"), 2)).first()[0])

In [0]:
# What is 381? Test candidate aggregations against the gold fact table.
f = spark.table("workspace.cyber_threat.gold_fact_vulnerability")

print("SUM(epss_score * 4):", f.agg(F.round(F.sum(F.col("epss_score") * 4), 1)).first()[0])
print("SUM(epss_score):", f.agg(F.round(F.sum("epss_score"), 2)).first()[0])
print("SUM(cvss_base_score):", f.agg(F.round(F.sum("cvss_base_score"), 1)).first()[0])
print("AVG risk per hostname (6,848/20):", round(6848.1 / 20, 1))

# per-host with old 10-weight for KEV rows, to see if any host = 381
alt = (f.join(spark.table("workspace.cyber_threat.gold_dim_asset"), on="asset_id")
    .withColumn("pts", F.when(F.col("is_in_kev") == True, F.lit(10.0))
        .otherwise(F.coalesce(F.col("epss_score"), F.lit(0.0)) * 4
                   + F.coalesce(F.col("cvss_base_score"), F.lit(0.0))))
    .groupBy("hostname").agg(F.round(F.sum("pts"), 1).alias("risk_w10"))
    .where(F.col("risk_w10").between(350, 420))
    .orderBy("risk_w10"))
print("\nHosts with 10-weighted risk in 350-420 range:")
alt.show()

In [0]:
# Ground truth for Power BI: composite Risk Score per os_name (same DAX formula).
risk_os = (spark.table("workspace.cyber_threat.gold_fact_vulnerability")
    .join(spark.table("workspace.cyber_threat.gold_dim_asset"), on="asset_id")
    .withColumn("risk_points",
        F.when(F.col("is_in_kev") == True, F.lit(15.0))
         .otherwise(F.coalesce(F.col("epss_score"), F.lit(0.0)) * 4
                    + F.coalesce(F.col("cvss_base_score"), F.lit(0.0))))
    .groupBy("os_name")
    .agg(F.round(F.sum("risk_points"), 1).alias("risk_score"),
         F.count("*").alias("exposures"),
         F.countDistinct("asset_id").alias("assets"))
    .orderBy(F.desc("risk_score")))
display(risk_os)

print("Check: total =", risk_os.agg(F.round(F.sum("risk_score"), 1)).first()[0])

In [0]:
# ===== BEFORE: gold table file layout (screenshot this) =====
for t in ["gold_dim_cve", "gold_fact_vulnerability"]:
    d = (spark.sql(f"DESCRIBE DETAIL workspace.cyber_threat.{t}")
         .select("numFiles", "sizeInBytes", "clusteringColumns")
         .toPandas())
    print(f"== {t} BEFORE ==")
    print(d.to_string(index=False))
    print()

In [0]:
# ===== Apply liquid clustering + OPTIMIZE gold_dim_cve (screenshot this) =====
# Cluster on published_date: the most common filter/analysis column for the CVE dim.
spark.sql("ALTER TABLE workspace.cyber_threat.gold_dim_cve CLUSTER BY (published_date)")

opt = spark.sql("OPTIMIZE workspace.cyber_threat.gold_dim_cve")
print("== OPTIMIZE metrics: gold_dim_cve ==")
opt.show(truncate=False)

d = (spark.sql("DESCRIBE DETAIL workspace.cyber_threat.gold_dim_cve")
     .select("numFiles", "sizeInBytes", "clusteringColumns")
     .toPandas())
print("== gold_dim_cve AFTER ==")
print(d.to_string(index=False))

In [0]:
# ===== OPTIMIZE the fact table too (compaction only, keeps its layout) =====
opt2 = spark.sql("OPTIMIZE workspace.cyber_threat.gold_fact_vulnerability")
print("== OPTIMIZE metrics: gold_fact_vulnerability ==")
opt2.show(truncate=False)

d2 = (spark.sql("DESCRIBE DETAIL workspace.cyber_threat.gold_fact_vulnerability")
      .select("numFiles", "sizeInBytes", "clusteringColumns")
      .toPandas())
print("== gold_fact_vulnerability AFTER ==")
print(d2.to_string(index=False))

In [0]:
# ===== Sanity: OPTIMIZE/CLUSTER BY changed layout only — data unchanged =====
for t in ["gold_dim_cve", "gold_fact_vulnerability", "gold_dim_asset", "gold_dim_date", "gold_dim_software", "gold_dim_severity"]:
    print(f"{t}: {spark.table(f'workspace.cyber_threat.{t}').count()} rows")

f = spark.table("workspace.cyber_threat.gold_fact_vulnerability")
print("\nFact integrity:",
      f.select("asset_id", "cve_key").distinct().count(), "distinct (asset_id, cve_key) pairs,",
      f.where("cve_key is null or asset_id is null").count(), "rows with null keys")

In [0]:
# Report screenshot: Silver-to-Gold execution history from the audit log table.
# (No Lakeflow Job is scheduled yet, so pipeline_execution_logs IS the run graph.)
log_result = spark.sql("""
    SELECT target_table, status, rows_inserted, rows_updated,
           round(unix_timestamp(end_time) - unix_timestamp(start_time), 1) AS duration_seconds,
           date_format(start_time, 'yyyy-MM-dd HH:mm:ss') AS started_at
    FROM workspace.cyber_threat.pipeline_execution_logs
    WHERE layer = 'Silver-to-Gold'
    ORDER BY start_time DESC
    LIMIT 40
""")
display(log_result)